# Visualisasi Hasil Ekspor GGUF (Bab 4.3)
Notebook ini menghasilkan seluruh grafik untuk subbab 4.3 (Hasil Ekspor Model ke Format GGUF dan Evaluasi — Tujuan 3), berdasarkan output `tujuan3/17-compute-automatic-eval-gguf.ipynb`, dengan pembanding hasil model safetensors dari Tujuan 2 (`tujuan2/[3] evaluation/11-compute-automatic-eval.ipynb`).

**Berkas wajib** (letakkan pada direktori yang sama dengan notebook ini):
- `summary_metrics_gguf.csv`
- `gemma2_base_scored_gguf.csv`, `gemma2_finetuned_scored_gguf.csv`
- `llama3.2_base_scored_gguf.csv`, `llama3.2_finetuned_scored_gguf.csv`

**Berkas opsional** (untuk grafik pembanding GGUF vs Safetensors — salin dari folder `Visualisasi` hasil Tujuan 2, atau unduh ulang dari HF repo sebelum `summary_metrics.csv` tertimpa oleh notebook 17):
- `summary_metrics.csv` (hasil safetensors Tujuan 2)
- `gemma2_base_scored.csv`, `gemma2_finetuned_scored.csv`, `llama3.2_base_scored.csv`, `llama3.2_finetuned_scored.csv` (hasil safetensors Tujuan 2)

## Setup, pemuatan data, dan konfigurasi global

In [ ]:
%matplotlib inline
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({"font.family": "serif", "font.size": 10, "axes.grid": True,
                     "grid.alpha": 0.3, "axes.axisbelow": True, "figure.dpi": 150})

MODEL_KEYS = ["gemma2", "llama3.2"]
CONDITIONS = ["base", "finetuned"]
METRICS = ["em", "f1", "rouge_l", "indobertscore"]
METRIC_LABELS = {"em": "Exact Match", "f1": "Token F1", "rouge_l": "ROUGE-L", "indobertscore": "IndoBERTScore"}
MODEL_LABELS = {"gemma2": "Gemma-2-2B", "llama3.2": "Llama-3.2-3B"}

C_BASE, C_FT = "#b3b3b3", "#2c6fbb"
C_SAFE, C_GGUF = "#3a7d44", "#e08a1e"
C_SIG = "#c0392b"

summary_gguf = pd.read_csv("summary_metrics_gguf.csv").set_index("model")

scored_gguf = {}
for mk in MODEL_KEYS:
    for cond in CONDITIONS:
        fname = f"{mk}_{cond}_scored_gguf.csv"
        if os.path.exists(fname):
            scored_gguf[(mk, cond)] = pd.read_csv(fname)
        else:
            print(f"[WARNING] {fname} tidak ditemukan — sebagian grafik akan dilewati.")

HAS_SAFETENSORS = os.path.exists("summary_metrics.csv")
if HAS_SAFETENSORS:
    summary_safe = pd.read_csv("summary_metrics.csv").set_index("model")
    scored_safe = {}
    for mk in MODEL_KEYS:
        for cond in CONDITIONS:
            fname = f"{mk}_{cond}_scored.csv"
            if os.path.exists(fname):
                scored_safe[(mk, cond)] = pd.read_csv(fname)
else:
    print("[INFO] summary_metrics.csv (safetensors, Tujuan 2) tidak ditemukan — "
          "grafik pembanding GGUF vs Safetensors akan dilewati.")

def paired_frame_gguf(model_key, metric):
    df_base = scored_gguf[(model_key, "base")][["record_id", metric]].rename(columns={metric: "base"})
    df_ft = scored_gguf[(model_key, "finetuned")][["record_id", metric]].rename(columns={metric: "finetuned"})
    merged = df_base.merge(df_ft, on="record_id")
    return merged.dropna(subset=["base", "finetuned"]) if metric == "em" else merged


## Fungsi pembuat grafik (satu sel per grafik)

### Fig 1 — Ringkasan metrik otomatis (GGUF): base vs fine-tuned

In [ ]:
def fig1_ringkasan_metrik_gguf():
    fig, axes = plt.subplots(1, len(MODEL_KEYS), figsize=(6.5 * len(MODEL_KEYS), 5), sharey=True)
    x = np.arange(len(METRICS)); w = 0.35
    for ax, mk in zip(axes, MODEL_KEYS):
        base_vals = [summary_gguf.loc[f"{mk}_base", m] for m in METRICS]
        ft_vals = [summary_gguf.loc[f"{mk}_finetuned", m] for m in METRICS]
        ax.bar(x - w/2, base_vals, w, label="Base", color=C_BASE, edgecolor="black", linewidth=.5)
        ax.bar(x + w/2, ft_vals, w, label="Fine-tuned", color=C_FT, edgecolor="black", linewidth=.5)
        for xi, v in zip(x - w/2, base_vals): ax.text(xi, v + .01, f"{v:.3f}", ha="center", fontsize=7.5)
        for xi, v in zip(x + w/2, ft_vals): ax.text(xi, v + .01, f"{v:.3f}", ha="center", fontsize=7.5)
        ax.set_xticks(x); ax.set_xticklabels([METRIC_LABELS[m] for m in METRICS], rotation=15, ha="right")
        ax.set_title(f"{MODEL_LABELS[mk]} (GGUF)"); ax.set_ylim(0, 1.0)
    axes[0].set_ylabel("Skor"); axes[-1].legend(fontsize=8, loc="upper right")
    fig.suptitle("Perbandingan Metrik Otomatis Model GGUF: Base vs Fine-tuned", y=1.02)
    plt.tight_layout(); plt.savefig("fig1_ringkasan_metrik_gguf.png", bbox_inches="tight"); plt.show()

fig1_ringkasan_metrik_gguf()


### Fig 2 — Peningkatan (delta) fine-tuning per metrik (GGUF)

In [ ]:
def fig2_delta_improvement_gguf():
    x = np.arange(len(METRICS)); w = 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, mk in enumerate(MODEL_KEYS):
        vals = [summary_gguf.loc[f"{mk}_finetuned", m] - summary_gguf.loc[f"{mk}_base", m] for m in METRICS]
        offset = (i - 0.5) * w
        col = C_SAFE if mk == "gemma2" else C_GGUF
        ax.bar(x + offset, vals, w, label=MODEL_LABELS[mk], color=col, edgecolor="black", linewidth=.5)
        for xi, v in zip(x + offset, vals):
            ax.text(xi, v + (0.005 if v >= 0 else -0.015), f"{v:+.3f}", ha="center", fontsize=7.5)
    ax.axhline(0, color="black", lw=.8)
    ax.set_xticks(x); ax.set_xticklabels([METRIC_LABELS[m] for m in METRICS])
    ax.set_ylabel("Delta Skor (Fine-tuned − Base)")
    ax.set_title("Peningkatan Setelah Instruction-Tuning Tetap Terjaga pada Model GGUF"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig("fig2_delta_improvement_gguf.png", bbox_inches="tight"); plt.show()

fig2_delta_improvement_gguf()


### Fig 3 — Dampak kuantisasi GGUF terhadap performa (GGUF vs Safetensors, opsional)

In [ ]:
def fig3_gguf_vs_safetensors():
    if not HAS_SAFETENSORS:
        print("Dilewati: summary_metrics.csv (safetensors) tidak ditemukan."); return
    fig, axes = plt.subplots(1, len(MODEL_KEYS), figsize=(6.5 * len(MODEL_KEYS), 5), sharey=True)
    x = np.arange(len(METRICS)); w = 0.2
    for ax, mk in zip(axes, MODEL_KEYS):
        safe_base = [summary_safe.loc[f"{mk}_base", m] for m in METRICS]
        safe_ft = [summary_safe.loc[f"{mk}_finetuned", m] for m in METRICS]
        gguf_base = [summary_gguf.loc[f"{mk}_base", m] for m in METRICS]
        gguf_ft = [summary_gguf.loc[f"{mk}_finetuned", m] for m in METRICS]
        ax.bar(x - 1.5*w, safe_base, w, color=C_BASE, edgecolor="black", linewidth=.5, label="Base (Safetensors)")
        ax.bar(x - 0.5*w, gguf_base, w, color=C_BASE, edgecolor="black", linewidth=.5, hatch="//", label="Base (GGUF)")
        ax.bar(x + 0.5*w, safe_ft, w, color=C_FT, edgecolor="black", linewidth=.5, label="Fine-tuned (Safetensors)")
        ax.bar(x + 1.5*w, gguf_ft, w, color=C_FT, edgecolor="black", linewidth=.5, hatch="//", label="Fine-tuned (GGUF)")
        ax.set_xticks(x); ax.set_xticklabels([METRIC_LABELS[m] for m in METRICS], rotation=15, ha="right")
        ax.set_title(MODEL_LABELS[mk]); ax.set_ylim(0, 1.0)
    axes[0].set_ylabel("Skor")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, 0), ncol=4)
    fig.suptitle("Dampak Kuantisasi GGUF terhadap Performa (dibanding Safetensors)", y=1.03)
    plt.tight_layout(); plt.savefig("fig3_gguf_vs_safetensors.png", bbox_inches="tight"); plt.show()

fig3_gguf_vs_safetensors()


### Fig 4 — Degradasi kuantisasi per soal (GGUF − Safetensors F1, opsional)

In [ ]:
def fig4_degradasi_per_soal():
    if not HAS_SAFETENSORS:
        print("Dilewati: data safetensors tidak ditemukan."); return
    fig, axes = plt.subplots(1, len(MODEL_KEYS), figsize=(6.5 * len(MODEL_KEYS), 5), sharey=True)
    for ax, mk in zip(axes, MODEL_KEYS):
        for cond, col in [("base", C_BASE), ("finetuned", C_FT)]:
            d_safe = scored_safe[(mk, cond)][["record_id", "f1"]].rename(columns={"f1": "f1_safe"})
            d_gguf = scored_gguf[(mk, cond)][["record_id", "f1"]].rename(columns={"f1": "f1_gguf"})
            d = d_safe.merge(d_gguf, on="record_id")
            diff = d["f1_gguf"] - d["f1_safe"]
            ax.hist(diff, bins=40, alpha=.55, color=col, label=cond.capitalize(), edgecolor="black", linewidth=.2)
        ax.axvline(0, color="black", lw=.8)
        ax.set_xlabel("Selisih F1 (GGUF − Safetensors)"); ax.set_title(MODEL_LABELS[mk]); ax.legend(fontsize=8)
    axes[0].set_ylabel("Jumlah Soal")
    fig.suptitle("Distribusi Selisih F1 per Soal Akibat Kuantisasi GGUF", y=1.02)
    plt.tight_layout(); plt.savefig("fig4_degradasi_per_soal.png", bbox_inches="tight"); plt.show()

fig4_degradasi_per_soal()


### Fig 5 — Sebaran skor F1 tingkat-soal (GGUF, boxplot)

In [ ]:
def fig5_boxplot_f1_gguf():
    labels = [f"{MODEL_LABELS[mk]}\n{c}" for mk in MODEL_KEYS for c in ("Base", "Fine-tuned")]
    data = [scored_gguf[(mk, cond)]["f1"].values for mk in MODEL_KEYS for cond in CONDITIONS]
    colors = [C_BASE, C_FT] * len(MODEL_KEYS)
    fig, ax = plt.subplots(figsize=(8, 5.5))
    bp = ax.boxplot(data, vert=True, patch_artist=True, widths=.6, showmeans=True)
    for patch, col in zip(bp["boxes"], colors):
        patch.set_facecolor(col); patch.set_alpha(.8)
    ax.set_xticklabels(labels, fontsize=8.5)
    ax.set_ylabel("Token F1 (per soal)")
    ax.set_title("Sebaran Skor F1 Tingkat-Soal (Model GGUF): Base vs Fine-tuned")
    plt.tight_layout(); plt.savefig("fig5_boxplot_f1_gguf.png", bbox_inches="tight"); plt.show()

fig5_boxplot_f1_gguf()


### Fig 6 — Menang/seri/kalah per soal (GGUF: fine-tuned vs base, F1)

In [ ]:
def fig6_win_rate_gguf():
    rows = []
    for mk in MODEL_KEYS:
        d = paired_frame_gguf(mk, "f1")
        menang = (d["finetuned"] > d["base"]).sum()
        seri = (d["finetuned"] == d["base"]).sum()
        kalah = (d["finetuned"] < d["base"]).sum()
        rows.append({"model": mk, "Menang": menang, "Seri": seri, "Kalah": kalah})
    d = pd.DataFrame(rows).set_index("model")
    fig, ax = plt.subplots(figsize=(7, 5))
    left = np.zeros(len(d))
    for cat, col in [("Menang", C_FT), ("Seri", C_BASE), ("Kalah", C_SIG)]:
        vals = d[cat].values
        ax.barh([MODEL_LABELS[m] for m in d.index], vals, left=left, label=cat, color=col, edgecolor="black", linewidth=.5)
        for y, (v, l) in enumerate(zip(vals, left)):
            if v > 0: ax.text(l + v/2, y, str(v), ha="center", va="center", fontsize=8)
        left += vals
    ax.set_xlabel("Jumlah Soal"); ax.set_title("Fine-tuned vs Base per Soal pada Model GGUF (Token F1)")
    ax.legend(fontsize=8, loc="lower right")
    plt.tight_layout(); plt.savefig("fig6_win_rate_gguf.png", bbox_inches="tight"); plt.show()

fig6_win_rate_gguf()


### Fig 7 — Signifikansi statistik (Wilcoxon berpasangan, GGUF) per model x metrik

In [ ]:
def fig7_heatmap_pvalue_gguf():
    pvals = np.zeros((len(MODEL_KEYS), len(METRICS)))
    for i, mk in enumerate(MODEL_KEYS):
        for j, m in enumerate(METRICS):
            d = paired_frame_gguf(mk, m)
            diff = d["finetuned"] - d["base"]
            if diff.abs().sum() == 0:
                pvals[i, j] = 1.0
            else:
                try:
                    _, p = stats.wilcoxon(diff)
                except ValueError:
                    p = np.nan
                pvals[i, j] = p
    fig, ax = plt.subplots(figsize=(7, 4))
    im = ax.imshow(pvals, cmap="RdYlGn_r", vmin=0, vmax=.1, aspect="auto")
    ax.set_xticks(range(len(METRICS))); ax.set_xticklabels([METRIC_LABELS[m] for m in METRICS], rotation=20, ha="right")
    ax.set_yticks(range(len(MODEL_KEYS))); ax.set_yticklabels([MODEL_LABELS[m] for m in MODEL_KEYS])
    for i in range(len(MODEL_KEYS)):
        for j in range(len(METRICS)):
            p = pvals[i, j]
            label = "p<0,001" if p < .001 else f"p={p:.3f}"
            mark = " *" if p < .05 else ""
            ax.text(j, i, label + mark, ha="center", va="center", fontsize=8,
                    color="white" if p < .05 else "black")
    cb = fig.colorbar(im, ax=ax, fraction=.03, pad=.02); cb.set_label("p-value (Wilcoxon signed-rank)")
    ax.set_title("Signifikansi Peningkatan Fine-tuning pada Model GGUF\n(* = signifikan, p<0,05)")
    plt.tight_layout(); plt.savefig("fig7_heatmap_pvalue_gguf.png", bbox_inches="tight"); plt.show()

fig7_heatmap_pvalue_gguf()


### Fig 8 — Ringkasan akhir: Safetensors vs GGUF, Base vs Fine-tuned (tabel + heatmap gabungan)

In [ ]:
def fig8_ringkasan_akhir():
    if not HAS_SAFETENSORS:
        print("Dilewati: data safetensors tidak ditemukan."); return
    rows = []
    for mk in MODEL_KEYS:
        for cond in CONDITIONS:
            rows.append({"model": MODEL_LABELS[mk], "condition": cond, "format": "Safetensors",
                         **{m: summary_safe.loc[f"{mk}_{cond}", m] for m in METRICS}})
            rows.append({"model": MODEL_LABELS[mk], "condition": cond, "format": "GGUF",
                         **{m: summary_gguf.loc[f"{mk}_{cond}", m] for m in METRICS}})
    d = pd.DataFrame(rows)
    mat = d.set_index(["model", "format", "condition"])[METRICS]

    fig, ax = plt.subplots(figsize=(8, 0.55 * len(mat) + 1.5))
    im = ax.imshow(mat.values, cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(METRICS))); ax.set_xticklabels([METRIC_LABELS[m] for m in METRICS], rotation=20, ha="right")
    ax.set_yticks(range(len(mat))); ax.set_yticklabels([f"{a} / {b} / {c}" for a, b, c in mat.index], fontsize=8)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat.values[i, j]
            ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=7.5, color="white" if v > .55 else "black")
    cb = fig.colorbar(im, ax=ax, fraction=.03, pad=.02); cb.set_label("Skor")
    ax.set_title("Ringkasan Akhir: Model x Format x Kondisi x Metrik")
    plt.tight_layout(); plt.savefig("fig8_ringkasan_akhir.png", bbox_inches="tight"); plt.show()
    return d

fig8_ringkasan_akhir()
